<a href="https://colab.research.google.com/github/cursuri-inf/IPAP/blob/main/saptamana-05/L5_bucla_completa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Caietul „Bucla completă și hiperparametrii” (Laborator 6)

Antrenați o rețea cu bucla completă din cursul 6: la fiecare epocă, pașii pe loturile de antrenare, apoi pierderea pe validare. Întrebarea rămâne „mascul sau femelă?”, de data aceasta pe toate cele patru măsurători și pe specie, cu o rețea de 32 de neuroni ascunși. Completați validarea din buclă, citiți curbele, alegeți optimizatorul și rata pe validare și rulați scriptul `train.py` cu alegerea dumneavoastră, de două ori, cu aceleași cifre. Nivelul de bază (secțiunile 0–4) se termină în laborator.

Pașii:

1. Deschideți caietul cu butonul „Open in Colab” de mai sus. La prima rulare, Colab avertizează că documentul nu a fost creat de Google: caietul vine din depozitul cursului, așa că alegeți „Run anyway”.
2. Înainte să scrieți ceva: File → Save a copy in Drive; lucrați în copie.
3. Lucrați în pereche cu un coleg. Pilotul rulează și scrie, navigatorul citește, prezice și verifică; schimbați rolurile la fiecare secțiune. Doar dacă numărul celor prezenți este impar, o grupă are 3 studenți, iar al treilea coleg este verificator: rulează în copia lui și compară rezultatele. Fiecare predă caietul lui.
4. Rulați celulele în ordine, cu Shift+Enter. La fiecare „Exercițiul B…” vă opriți și scrieți răspunsul în celula „Răspunsul dumneavoastră:” de sub el sau completați codul, acolo unde vedeți `...` sau `None`.
5. La final: Runtime → Restart session and run all (rularea de la zero), verificați că ultima celulă de cod afișează „Caietul a rulat până la capăt”, completați „Declarația AI”, descărcați caietul (File → Download → Download .ipynb) și îl încărcați la activitatea L6 din Campus Virtual.

Tot caietul rulează pe procesor (CPU), în aproximativ un minut; nu aveți nevoie de GPU. Extensiile E1–E3 sunt opționale, pentru nota 8–10.

Completați:

- Numele dumneavoastră: …
- Colegul de pereche: …

## Planul meu pentru săptămânile 7–8

Punctul de control al traseelor: îl completați dacă sunteți pe traseul Rampă sau Nivelare Python ori dacă nu ați terminat nivelul de bază la laboratorul 4 sau 5; pentru ceilalți este opțional. Nu se notează. Titularul trece pe la fiecare, în timpul laboratorului, câte trei minute.

- Ce mi-a fost greu până acum: …
- Ce recuperez și până când: …
- De ce ajutor am nevoie: …

## Harta caietului

Fiecare secțiune pune la lucru un capitol din suportul cursului 6.

| Secțiunea | Capitolul din suportul cursului 6 |
|---|---|
| 1. Bucla completă | 2, Bucla completă |
| 2. Citirea curbelor | 3, Citirea curbelor |
| 3. Optimizatorul și rata | 4, Optimizatorii |
| 4. Scriptul de antrenare | 5, Scriptul de antrenare |
| Extensia E1 | 4, Optimizatorii (mărimea lotului) |
| Extensia E2 | 4.3, Adam („Pentru curioși”) |
| Extensia E3 | 5.3, Scriptul la proiect |

## 0. Pornire

Încărcați bibliotecile și datele și pregătiți rețeaua. Datele: cei 333 de pinguini pentru care se știe dacă sunt masculi sau femele, împărțiți 70/15/15, stratificat pe specii, ca la laboratorul 5. Intrările sunt șapte: cele patru măsurători (lungimea și adâncimea ciocului, lungimea aripii, masa corporală), standardizate pe antrenare, și specia, scrisă ca trei coloane cu 0 și 1, câte una pentru fiecare specie. Eticheta este 1 pentru mascul și 0 pentru femelă (în tabel, coloana `sex`).

Dacă adresa datelor nu răspunde, descărcați fișierul `penguins.csv` din Campus Virtual (Cursul 3, „Datele cu pinguini”), îl încărcați în panoul Files din bara din stânga și rulați din nou celula cu datele.

In [ ]:
import math, os, sys, time
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.optim import SGD, Adam
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

T_START = time.time()  # pentru durata totală, afișată la final

print("Python ", sys.version.split()[0])
print("PyTorch", torch.__version__)


def procent(x):
    """0.8612 -> '86,1%' (cu virgulă zecimală)."""
    return f"{100 * x:.1f}%".replace(".", ",")


# culorile graficelor cursului
TEXT, SECUNDAR, ACCENT, SERIE2 = "#161616", "#4A4A4A", "#9B1B30", "#4E6375"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": False,
                     "text.color": TEXT, "axes.labelcolor": SECUNDAR,
                     "xtick.color": SECUNDAR, "ytick.color": SECUNDAR})

In [ ]:
URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
MASURI = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

sursa = "penguins.csv" if os.path.exists("penguins.csv") else URL
mf = pd.read_csv(sursa).dropna(subset=MASURI + ["species", "sex"])  # 333 de pinguini
rest, test = train_test_split(mf, test_size=0.15, stratify=mf["species"], random_state=0)
ant, val = train_test_split(rest, test_size=0.15 / 0.85, stratify=rest["species"], random_state=0)
MED, AB = ant[MASURI].mean(), ant[MASURI].std()
SPECII = pd.get_dummies(ant[["species"]].astype(str)).columns  # o coloană 0/1 pentru fiecare specie


def tensori(t):
    """Cele patru măsurători standardizate pe antrenare, specia (trei coloane 0/1) și eticheta (1 pentru mascul)."""
    x = pd.concat([(t[MASURI] - MED) / AB,
                   pd.get_dummies(t[["species"]].astype(str)).reindex(columns=SPECII, fill_value=0)], axis=1)
    y = (t["sex"].astype(str) == "MALE").values
    return torch.tensor(x.values.astype("float32")), torch.tensor(y).float().reshape(-1, 1)


z_ant, y_ant = tensori(ant)
z_val, y_val = tensori(val)
REFERINTA = 1 - y_val.mean().item()  # mereu femelă
print("antrenare", len(ant), "| validare", len(val), "| test", len(test), "| intrări:", z_ant.shape[1])
print("referința (mereu femelă) pe validare:", procent(REFERINTA))

In [ ]:
def pas(model, pierdere_f, opt, zb, yb):
    """Un pas al buclei, pe un lot (funcția din laboratorul 4)."""
    predictie = model(zb)
    pierdere = pierdere_f(predictie, yb)
    opt.zero_grad()
    pierdere.backward()
    opt.step()


def retea_noua(ascunsi=32):
    """7 intrări, un strat ascuns cu ReLU, un neuron de ieșire; de fiecare dată aceleași valori de pornire."""
    torch.manual_seed(0)
    return nn.Sequential(nn.Linear(7, ascunsi), nn.ReLU(), nn.Linear(ascunsi, 1))


def optimizator(nume, model, rata):
    """"sgd", "moment" (SGD cu momentul 0,9) sau "adam", ca în train.py."""
    if nume == "adam":
        return Adam(model.parameters(), lr=rata)
    return SGD(model.parameters(), lr=rata, momentum=0.9 if nume == "moment" else 0)


print("parametri:", sum(p.numel() for p in retea_noua().parameters()))

## 1. Bucla completă

La sfârșitul fiecărei epoci, bucla completă măsoară pierderea și acuratețea pe tot setul de antrenare și pe tot setul de validare (cursul 6, §2.1). Măsurarea se face în funcția `evalueaza`, care nu are voie să schimbe nimic în model.

### Exercițiul B1 (completați)

Completați cele trei rânduri marcate cu B1 din `evalueaza`:

1. B1.1: trece modelul în modul de evaluare;
2. B1.2: calculul de sub `with` trebuie să se facă fără gradient (acum se face cu gradient);
3. B1.3: pierderea pe tot setul, ca număr Python (nu ca tensor).

Celula de după verifică funcția și vă spune ce mai lipsește. Până la „B1: corect”, secțiunile 1–3 nu rulează antrenările.

In [ ]:
def evalueaza(model, pf, z, y):
    """Pierderea și acuratețea modelului pe un set întreg, fără să învețe nimic din el."""
    ...                          # B1.1: trece modelul în modul de evaluare
    with torch.enable_grad():    # B1.2: aici calculul se face cu gradient; faceți-l fără
        scor = model(z)
        pierdere = None          # B1.3: pierderea pe tot setul, ca număr Python
        corect = ((scor > 0).float() == y).float().mean().item()
    model.train()                # înapoi la antrenare
    return pierdere, corect


def antreneaza(model, opt, epoci, lot=32):
    """Bucla completă: la fiecare epocă, pașii pe loturi, apoi pierderea și acuratețea pe antrenare și pe validare."""
    pf = nn.BCEWithLogitsLoss()
    loturi = DataLoader(TensorDataset(z_ant, y_ant), batch_size=lot, shuffle=True)
    istoric = []
    for epoca in range(epoci):
        for zb, yb in loturi:
            pas(model, pf, opt, zb, yb)
        istoric.append(evalueaza(model, pf, z_ant, y_ant) + evalueaza(model, pf, z_val, y_val))
    return istoric  # pe fiecare rând: pierderea și acuratețea pe antrenare, apoi pe validare

In [ ]:
class _Sonda(nn.Module):
    """Pusă în fața rețelei, ține minte în ce mod a fost rulată și dacă se calcula gradientul."""
    def forward(self, x):
        self.mod_antrenare, self.cu_gradient = self.training, torch.is_grad_enabled()
        return x


def verifica_b1():
    sonda = _Sonda()
    model = nn.Sequential(sonda, retea_noua())
    pf = nn.BCEWithLogitsLoss()
    with torch.no_grad():
        corect_p = pf(model(z_val), y_val).item()
    try:
        p, a = evalueaza(model, pf, z_val, y_val)
    except Exception as e:
        if "context manager" in str(e):
            return "B1: nu încă. Blocul fără gradient se scrie cu paranteze: with torch.no_grad():"
        return f"B1: nu încă. Funcția dă o eroare: {type(e).__name__}: {str(e).splitlines()[0]}"
    if p is None:
        return "B1: nu încă. Completați cele trei rânduri marcate cu B1 din evalueaza."
    if sonda.mod_antrenare:
        return "B1: nu încă. Modelul nu trece în modul de evaluare (B1.1, §2.1)."
    if sonda.cu_gradient:
        return "B1: nu încă. Calculul se face încă cu gradient: blocul de la B1.2 trebuie să fie fără gradient (§2.1)."
    if not isinstance(p, float):
        return "B1: nu încă. Pierderea trebuie să fie un număr Python, nu un tensor: adăugați .item() la sfârșit (B1.3)."
    if abs(p - corect_p) > 1e-6:
        return "B1: nu încă. Pierderea nu este cea pe tot setul primit: pf(scor, y) (B1.3)."
    if not model.training:
        return "B1: nu încă. La sfârșit, modelul trebuie readus în modul de antrenare (model.train())."
    return "B1: corect."


mesaj = verifica_b1()
B1_CORECT = mesaj == "B1: corect."
print(mesaj)

Rețeaua, antrenată cu SGD și rata 0,1, 300 de epoci, ca în curs. Afișați patru rânduri din istoric și cele două curbe.

In [ ]:
def curbe(ax, ist, titlu, epoca_aleasa=None):
    """Pierderea pe antrenare (negru) și pe validare (carmin), epocă cu epocă."""
    e = range(1, len(ist) + 1)
    ax.plot(e, [h[0] for h in ist], color=TEXT, lw=1.4, label="antrenare")
    ax.plot(e, [h[2] for h in ist], color=ACCENT, lw=1.8, label="validare")
    if epoca_aleasa:
        ax.plot(epoca_aleasa, ist[epoca_aleasa - 1][2], "o", color=ACCENT)
    ax.set_xlabel("epoca")
    ax.set_ylabel("pierderea")
    ax.set_title(titlu, loc="left")
    ax.legend(frameon=False)


def cea_mai_buna(ist):
    """Epoca (de la 1) cu pierderea cea mai mică pe validare."""
    return min(range(len(ist)), key=lambda i: ist[i][2]) + 1


if B1_CORECT:
    r = retea_noua()
    ist_sgd = antreneaza(r, optimizator("sgd", r, 0.1), 300)
    print("epoca | pierderea: antrenare, validare | acuratețea: antrenare, validare")
    for e in (1, 10, 100, 300):
        h = ist_sgd[e - 1]
        print(f"{e:5} | {h[0]:.3f}  {h[2]:.3f} | {procent(h[1]):>6}  {procent(h[3]):>6}")
    fig, ax = plt.subplots(figsize=(7, 3.5))
    curbe(ax, ist_sgd, "SGD, rata 0,1")
    plt.show()
else:
    print("Secțiunea 1 rulează după ce B1 este corect.")

## 2. Citirea curbelor

Aceeași rețea, cu Adam și rata 0,01, 300 de epoci. Celula afișează câteva rânduri din istoric, epoca cu pierderea cea mai mică pe validare (punctul de pe grafic) și curbele.

### Exercițiul B2 (interpretați)

1. La ce epocă are pierderea pe validare minimul? Ce epocă alegeți pentru acest model și pe ce set ați ales-o?
2. Ce se întâmplă după acea epocă pe cele două curbe? Cum se numește ce vedeți (cursul 6, §3.3)?
3. Acuratețea pe validare la epoca aleasă și la epoca 300: ce spune diferența dintre ele, față de diferența dintre pierderi?

In [ ]:
if B1_CORECT:
    r = retea_noua()
    ist_adam = antreneaza(r, optimizator("adam", r, 0.01), 300)
    e_min = cea_mai_buna(ist_adam)
    print("epoca | pierderea: antrenare, validare | acuratețea: antrenare, validare")
    for e in sorted({1, 10, e_min, 150, 300}):
        h = ist_adam[e - 1]
        semn = "  <- minimul pe validare" if e == e_min else ""
        print(f"{e:5} | {h[0]:.3f}  {h[2]:.3f} | {procent(h[1]):>6}  {procent(h[3]):>6}{semn}")
    fig, ax = plt.subplots(figsize=(7, 3.5))
    curbe(ax, ist_adam, "Adam, rata 0,01", epoca_aleasa=e_min)
    plt.show()
else:
    print("Secțiunea 2 rulează după ce B1 este corect.")

Răspunsul dumneavoastră:

## 3. Optimizatorul și rata

Funcția `incearca` antrenează rețeaua de la aceleași valori de pornire, cu optimizatorul și rata date, și întoarce epoca cu pierderea cea mai mică pe validare, pierderea și acuratețea de la acea epocă. Pentru comparație: prima epocă la care pierderea pe validare coboară sub 0,30 (sau „–”, dacă nu coboară în 300 de epoci).

### Exercițiul B3 (modificați și notați)

1. Rulați celula și completați tabelul de mai jos pentru cele cinci configurații.
2. Adăugați în listă o configurație aleasă de dumneavoastră (de exemplu `("moment", 0.01)` sau `("adam", 0.003)`), rulați din nou și completați ultimul rând.
3. Ce optimizator, ce rată și câte epoci alegeți și de ce? (2–3 fraze; alegerea se face pe validare. Setul de validare are 50 de pinguini: la acuratețe, un pinguin înseamnă 2%, deci comparați mai ales pierderile.) Scrieți alegerea în celula de la secțiunea 4.

In [ ]:
def incearca(nume, rata, epoci=300):
    r = retea_noua()
    ist = antreneaza(r, optimizator(nume, r, rata), epoci)
    e = cea_mai_buna(ist)
    sub = next((k + 1 for k, h in enumerate(ist) if h[2] < 0.30), None)
    return e, ist[e - 1][2], ist[e - 1][3], sub


CONFIGURATII = [("sgd", 0.1), ("moment", 0.1), ("adam", 0.001), ("adam", 0.01), ("adam", 0.1)]
# Exercițiul B3: adăugați în listă o configurație aleasă de dumneavoastră, de exemplu ("moment", 0.01)

if B1_CORECT:
    print("optimizatorul, rata | minimul pe validare: epoca, pierderea, acuratețea | sub 0,30 la epoca")
    for nume, rata in CONFIGURATII:
        e, p, a, sub = incearca(nume, rata)
        print(f"{nume:>7} {rata:<6} | {e:4}  {p:.3f}  {procent(a):>6} | {sub if sub else '–':>4}")
else:
    print("Secțiunea 3 rulează după ce B1 este corect.")

Răspunsul dumneavoastră:

| Optimizatorul | Rata | Epoca minimului | Pierderea pe validare | Acuratețea pe validare | Sub 0,30 la epoca |
|---|---|---|---|---|---|
| SGD | 0,1 | | | | |
| moment | 0,1 | | | | |
| Adam | 0,001 | | | | |
| Adam | 0,01 | | | | |
| Adam | 0,1 | | | | |
| configurația dumneavoastră: … | | | | | |

Ce alegeți și de ce:

## 4. Scriptul de antrenare

Celula de mai jos scrie scriptul `train.py` în folderul caietului (`%%writefile`, cursul 6, §5.1). Este bucla completă din secțiunea 1, cu aceleași date și aceeași rețea, cu hiperparametrii ca argumente. Nu schimbați nimic în el; îl rulați cu argumente. Argumentul `--test` evaluează modelul de la ultima epocă și pe setul de test, o singură dată.

In [ ]:
%%writefile train.py
"""Antrenează o rețea cu un strat ascuns pentru o întrebare cu două răspunsuri (laboratorul 6).
Implicit: „mascul sau femelă?” pe pinguini, cu cele patru măsurători și specia.
Exemplu: python train.py --opt adam --rata 0.01 --epoci 80
Cu --test, modelul de la ultima epocă se evaluează și pe test (o singură dată, la final)."""
import argparse, os
import pandas as pd
import torch
from torch import nn
from torch.optim import SGD, Adam
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
MASURI = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

p = argparse.ArgumentParser(description=__doc__)
p.add_argument("--opt", default="sgd", choices=["sgd", "moment", "adam"])
p.add_argument("--rata", type=float, default=0.1)
p.add_argument("--epoci", type=int, default=300)
p.add_argument("--lot", type=int, default=32)
p.add_argument("--ascunsi", type=int, default=32)
p.add_argument("--samanta", type=int, default=0)
p.add_argument("--test", action="store_true", help="evaluează și pe test, la final")
# datele: implicit pinguinii; pentru datele dumneavoastră (extensia E3) se schimbă toate cinci
p.add_argument("--fisier", default="penguins.csv")
p.add_argument("--tinta", default="sex", help="coloana de prezis, cu exact două valori")
p.add_argument("--pozitiv", default="MALE", help="valoarea țintei care primește eticheta 1")
p.add_argument("--intrari", nargs="+", default=MASURI, help="coloanele numerice")
p.add_argument("--categorii", nargs="*", default=["species"], help="coloanele cu categorii (pot lipsi)")
a = p.parse_args()

# datele: împărțirea 70/15/15, stratificată pe prima coloană cu categorii (sau pe țintă)
sursa = a.fisier
if not os.path.exists(sursa) and a.fisier == "penguins.csv":
    sursa = URL
d = pd.read_csv(sursa).dropna(subset=a.intrari + a.categorii + [a.tinta])
valori = set(d[a.tinta].astype(str))
assert len(valori) == 2, f"Ținta are {len(valori)} valori; trebuie să aibă exact două."
assert a.pozitiv in valori, f"--pozitiv trebuie să fie una dintre valorile țintei: {sorted(valori)}"
strat = a.categorii[0] if a.categorii else a.tinta
rest, test = train_test_split(d, test_size=0.15, stratify=d[strat], random_state=0)
ant, val = train_test_split(rest, test_size=0.15 / 0.85, stratify=rest[strat], random_state=0)
med, ab = ant[a.intrari].mean(), ant[a.intrari].std()
cat = pd.get_dummies(ant[a.categorii].astype(str)).columns if a.categorii else []


def tensori(t):
    x = (t[a.intrari] - med) / ab  # standardizate pe antrenare
    if a.categorii:  # fiecare categorie devine o coloană cu 0 și 1
        x = pd.concat([x, pd.get_dummies(t[a.categorii].astype(str)).reindex(columns=cat, fill_value=0)], axis=1)
    y = (t[a.tinta].astype(str) == a.pozitiv).values
    return torch.tensor(x.values.astype("float32")), torch.tensor(y).float().reshape(-1, 1)


z_ant, y_ant = tensori(ant)
z_val, y_val = tensori(val)
def referinta(y):  # acuratețea celui care răspunde mereu cu clasa mai frecventă în set
    return max(y.mean().item(), 1 - y.mean().item())


print(f"date: {len(ant)} antrenare, {len(val)} validare, {len(test)} test, {z_ant.shape[1]} intrări"
      f" | referința pe validare: {referinta(y_val):.0%}")


def evalueaza(model, pf, z, y):
    model.eval()
    with torch.no_grad():
        scor = model(z)
        pierdere = pf(scor, y).item()
        corect = ((scor > 0).float() == y).float().mean().item()
    model.train()
    return pierdere, corect


# modelul și optimizatorul, după argumente; sămânța fixează valorile de pornire și ordinea loturilor
torch.manual_seed(a.samanta)
model = nn.Sequential(nn.Linear(z_ant.shape[1], a.ascunsi), nn.ReLU(), nn.Linear(a.ascunsi, 1))
if a.opt == "adam":
    opt = Adam(model.parameters(), lr=a.rata)
else:
    opt = SGD(model.parameters(), lr=a.rata, momentum=0.9 if a.opt == "moment" else 0)
pf = nn.BCEWithLogitsLoss()
loturi = DataLoader(TensorDataset(z_ant, y_ant), batch_size=a.lot, shuffle=True)

# bucla completă: antrenare, apoi validare
istoric = []
for epoca in range(1, a.epoci + 1):
    for zb, yb in loturi:
        pierdere = pf(model(zb), yb)
        opt.zero_grad()
        pierdere.backward()
        opt.step()
    h = evalueaza(model, pf, z_ant, y_ant) + evalueaza(model, pf, z_val, y_val)
    istoric.append(h)
    if epoca % max(1, a.epoci // 5) == 0:
        print(f"epoca {epoca:4}: pierderea {h[0]:.3f} / {h[2]:.3f} | validare {h[3]:.0%}")

e = min(range(a.epoci), key=lambda i: istoric[i][2])
print(f"minimul pe validare: {istoric[e][2]:.3f}, la epoca {e + 1} ({istoric[e][3]:.0%})")
if a.test:
    z_test, y_test = tensori(test)
    _, acc = evalueaza(model, pf, z_test, y_test)
    print(f"pe test, modelul de la epoca {a.epoci}: {acc:.0%} | referința pe test: {referinta(y_test):.0%}")

### Exercițiul B4 (rulați și interpretați)

Scrieți în celula de mai jos alegerea de la B3: optimizatorul (`"sgd"`, `"moment"` sau `"adam"`), rata și numărul de epoci (epoca minimului pe validare). Celula rulează `train.py` de două ori, cu aceeași comandă, și compară cele două rulări. Celula de după rulează o singură dată, cu `--test`.

1. Ies cifrele celor două rulări identice? De ce (cursul 6, §5.2)?
2. Cât dă modelul pe test, față de validare și față de referință? De ce pot diferi cifra de pe validare și cea de pe test?
3. De ce rulați cu `--test` o singură dată, abia după ce ați ales optimizatorul, rata și epocile?

In [ ]:
OPT_ALES, RATA_ALEASA, EPOCI_ALESE = None, None, None  # Exercițiul B4: de exemplu "adam", 0.01, 80

RULARE_1 = RULARE_2 = None
if OPT_ALES is None:
    print("B4: scrieți întâi alegerea de la B3 în OPT_ALES, RATA_ALEASA și EPOCI_ALESE.")
elif OPT_ALES not in ("sgd", "moment", "adam"):
    print('B4: optimizatorul se scrie cu litere mici, între ghilimele: "sgd", "moment" sau "adam".')
else:
    COMANDA = f"python train.py --opt {OPT_ALES} --rata {RATA_ALEASA} --epoci {EPOCI_ALESE}"
    print(COMANDA, "(prima rulare)")
    RULARE_1 = !{COMANDA}
    print(*RULARE_1, sep="\n")
    print("\n" + COMANDA, "(a doua rulare)")
    RULARE_2 = !{COMANDA}
    print(*RULARE_2, sep="\n")
    print("\nAceleași cifre la ambele rulări:", "da" if list(RULARE_1) == list(RULARE_2) else "NU")

In [ ]:
TEST = None
if RULARE_1 is None:
    print("B4: rulați testul după ce ați scris alegerea în celula de mai sus.")
else:
    print(COMANDA + " --test")
    TEST = !{COMANDA} --test
    print(*TEST, sep="\n")

Răspunsul dumneavoastră:

## Extensii (opționale, pentru nota 8–10)

Alegeți cel puțin una. Fiecare are un comutator: îl puneți pe `True`, rulați celula și scrieți, în celula „Comentariul dumneavoastră” de sub ea, ce ați schimbat, ce rezultat ați obținut și cum îl explicați. Extensia pe care o predați rămâne cu comutatorul pe `True`, ca rezultatele ei să se vadă în caiet. Rulați extensiile după ce nivelul de bază este gata.

### Extensia E1: mărimea lotului

Aceeași rețea, cu SGD și rata 0,1, 100 de epoci, cu loturi de 8, 32, 128 și cu tot setul de antrenare într-un singur lot (233). Pentru fiecare: câți pași face optimizatorul, cât durează, minimul pe validare și pierderea la sfârșit; graficul pune cele patru curbe de validare una lângă alta. Ce câștigați și ce pierdeți cu un lot mai mic? Ce s-ar schimba dacă ați compara la același număr de pași, nu la același număr de epoci?

In [ ]:
RULEAZA_E1 = False  # True: rulați extensia

if RULEAZA_E1 and not B1_CORECT:
    print("E1 rulează după ce B1 este corect.")
elif RULEAZA_E1:
    fig, ax = plt.subplots(figsize=(7, 3.5))
    for lot, culoare in ((8, SERIE2), (32, TEXT), (128, SECUNDAR), (len(z_ant), ACCENT)):
        r = retea_noua()
        t0 = time.time()
        ist = antreneaza(r, optimizator("sgd", r, 0.1), 100, lot=lot)
        durata = time.time() - t0
        pasi = 100 * math.ceil(len(z_ant) / lot)
        e = cea_mai_buna(ist)
        print(f"lotul {lot:3}: {pasi:5} pași, {durata:4.1f} s | minimul pe validare {ist[e - 1][2]:.3f}"
              f" la epoca {e:3} | la sfârșit {ist[-1][2]:.3f}")
        ax.plot(range(1, 101), [h[2] for h in ist], color=culoare, lw=1.5, label=f"lotul {lot}")
    ax.set_xlabel("epoca")
    ax.set_ylabel("pierderea pe validare")
    ax.legend(frameon=False)
    plt.show()
else:
    print("E1 nu rulează (RULEAZA_E1 = False).")

Comentariul dumneavoastră (E1):

### Extensia E2: Adam scris de mână

Ca pasul lui SGD scris de mână la laboratorul 4, aici scrieți regula lui Adam (suportul cursului 6, §4.3, „Pentru curioși”). Clasa `AdamDeMana` are tot ce îi trebuie, în afară de cele două medii: completați cele două rânduri cu `...` din `step`. Apoi antrenați două rețele identice, 20 de epoci, una cu `AdamDeMana`, alta cu `torch.optim.Adam`, și comparați parametrii și pierderea pe validare. Ce observați și de ce?

In [ ]:
RULEAZA_E2 = False  # True: rulați extensia

if RULEAZA_E2:
    class AdamDeMana:
        """Regula lui Adam, pentru toți parametrii unui model."""
        def __init__(self, parametri, lr, b1=0.9, b2=0.999, eps=1e-8):
            self.p = list(parametri)
            self.lr, self.b1, self.b2, self.eps, self.t = lr, b1, b2, eps, 0
            self.m = [torch.zeros_like(w) for w in self.p]  # media gradienților
            self.s = [torch.zeros_like(w) for w in self.p]  # media pătratelor gradienților

        def zero_grad(self):
            for w in self.p:
                w.grad = None

        @torch.no_grad()
        def step(self):
            self.t += 1
            for w, m, s in zip(self.p, self.m, self.s):
                g = w.grad
                m.copy_(...)  # completați: m = 0,9 × m + 0,1 × g
                s.copy_(...)  # completați: s = 0,999 × s + 0,001 × g²
                m_cor = m / (1 - self.b1 ** self.t)  # corecția pentru primii pași
                s_cor = s / (1 - self.b2 ** self.t)
                w -= self.lr * m_cor / (s_cor.sqrt() + self.eps)

    try:
        r1 = retea_noua()  # sămânța fixată: aceleași valori de pornire și aceeași ordine a loturilor
        ist1 = antreneaza(r1, AdamDeMana(r1.parameters(), lr=0.01), 20)
        r2 = retea_noua()
        ist2 = antreneaza(r2, Adam(r2.parameters(), lr=0.01), 20)
    except TypeError:
        print("E2: completați întâi cele două rânduri cu ... din step.")
    else:
        dif = max((a - b).abs().max().item() for a, b in zip(r1.parameters(), r2.parameters()))
        print("cea mai mare diferență între parametri:", f"{dif:.1e}")
        if B1_CORECT:
            print("pierderea pe validare după 20 de epoci:", f"{ist1[-1][2]:.4f} (de mână),",
                  f"{ist2[-1][2]:.4f} (torch.optim.Adam)")
else:
    print("E2 nu rulează (RULEAZA_E2 = False).")

Comentariul dumneavoastră (E2):

### Extensia E3: datele dumneavoastră, cu `train.py`

Același script, pe un tabel ales de dumneavoastră, ca pregătire pentru P1: un fișier CSV cu o coloană de prezis care are exact două valori (ținta), câteva coloane numerice și, dacă există, coloane cu categorii. Utilizați doar date publice (de exemplu de pe data.gov.ro sau Kaggle) sau date generate de dumneavoastră; nu utilizați date personale și nici date interne de la locul de muncă.

Încărcați fișierul în panoul Files din bara din stânga sau puneți `INCARCA_CSV = True` și îl alegeți. Completați numele fișierului, ținta, valoarea care primește eticheta 1 și coloanele (separate prin spațiu, fără spații în numele lor). Scriptul împarte datele 70/15/15, standardizează coloanele numerice pe antrenare, antrenează rețeaua cu Adam și rata 0,01, 100 de epoci, și afișează referința simplă (clasa mai frecventă) pe validare. După ce ați încărcat fișierul, puneți înapoi `INCARCA_CSV = False`: altfel, la „Run all”, caietul se oprește și așteaptă un fișier.

In [ ]:
RULEAZA_E3 = False   # True: rulați extensia
INCARCA_CSV = False  # True: alegeți fișierul de pe calculator (doar în Colab)
FISIER_E3 = "datele_noastre.csv"
TINTA = "..."        # coloana de prezis, cu exact două valori
POZITIV = "..."      # valoarea țintei care primește eticheta 1
NUMERICE = "..."     # coloanele numerice, separate prin spațiu
CATEGORII = ""       # coloanele cu categorii, separate prin spațiu (pot lipsi)

if INCARCA_CSV:
    from google.colab import files  # există doar în Colab
    files.upload()

if not RULEAZA_E3:
    print("E3 nu rulează (RULEAZA_E3 = False).")
elif not os.path.exists(FISIER_E3):
    print(f"E3 nu rulează: fișierul {FISIER_E3} nu există.")
else:
    !python train.py --fisier {FISIER_E3} --tinta {TINTA} --pozitiv {POZITIV} --intrari {NUMERICE} --categorii {CATEGORII} --opt adam --rata 0.01 --epoci 100

Comentariul dumneavoastră (E3): ce date ați ales și de unde, ce ați prezis, cum se compară rețeaua cu referința simplă și ce arată curbele (minimul pe validare și epoca lui).

## Verificarea finală

Celula de mai jos are sens după Runtime → Restart session and run all: dacă afișează mesajul, caietul rulează de la început până la sfârșit fără erori.

In [ ]:
durata = time.time() - T_START
print(f"Caietul a rulat până la capăt (durata: {durata:.0f} s).")
print("referința (mereu femelă) pe validare:", procent(REFERINTA))
if not B1_CORECT:
    print("Atenție: B1 nu este încă corect; secțiunile 1–3 nu au rulat antrenările.")
if RULARE_1 is None:
    print("Atenție: B4 nu are încă alegerea de la B3 (OPT_ALES, RATA_ALEASA, EPOCI_ALESE).")
elif list(RULARE_1) != list(RULARE_2):
    print("Atenție: cele două rulări ale scriptului nu au dat aceleași cifre.")
if TEST is None:
    print("Atenție: B4 nu are încă rularea cu --test.")

## Ce ați învățat

- Bucla completă măsoară, la fiecare epocă, pierderea pe antrenare și pe validare, fără gradient și în modul de evaluare.
- Pe curbe se vede când antrenarea merge prea încet, când rata e prea mare și când modelul începe să învețe pe de rost: pierderea pe validare urcă, iar cea pe antrenare coboară.
- Optimizatorul, rata și numărul de epoci se aleg pe validare; testul se folosește o singură dată, la final.
- Un script cu sămânța fixată dă aceleași cifre la fiecare rulare, iar hiperparametrii se schimbă din comandă.

## Legătura cu cursul 7

La cursul 7 vedeți cum se împiedică supraînvățarea pe care ați văzut-o la B2 (regularizarea), pe date tabulare. În săptămâna 7 predați P1, propunerea de proiect; scriptul `train.py` este punctul de pornire pentru modelul dumneavoastră.

## Declarație AI

Completați la fiecare caiet predat, la persoana întâi:

- Instrumentul AI: …
- Pentru ce l-am folosit: …
- Ce am verificat singur (și cum): …

Dacă nu ați folosit niciun instrument AI, înlocuiți cele trei rânduri cu: Nu am folosit asistenți AI.